In [8]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score

# ==========================================
# 1. LOAD & PREPARE COMMON DATA
# ==========================================
print("Loading and preparing data...")

# Load
df_target = pd.read_csv('futu_margin_ratios_all_target.csv')
df_basic = pd.read_csv('futu_us_stock_basic_info.csv')
df_history = pd.read_csv('full_market_history_raw.csv')
df_shares = pd.read_csv('circulating_shares_report.csv')
df_map = pd.read_csv('LB_futu_mapping.csv')

# --- A. Bridge Keys (Mapping) ---
lb_to_futu_dict = dict(zip(df_map['LB_ticker'], df_map['futu_ticker']))
def get_futu_key(lb_symbol):
    if lb_symbol in lb_to_futu_dict: return lb_to_futu_dict[lb_symbol]
    if str(lb_symbol).endswith('.US'): return f"US.{str(lb_symbol).replace('.US', '')}"
    return f"US.{lb_symbol}"

df_history['futu_key'] = df_history['Symbol'].apply(get_futu_key)
df_shares['futu_key'] = df_shares['Symbol'].apply(get_futu_key)

# --- B. Basic Features (Turnover & Cap) ---
ANCHOR_DATE = pd.Timestamp("2026-01-20")
df_hist_filt = df_history[pd.to_datetime(df_history['Date']) < ANCHOR_DATE].copy()
df_hist_filt = df_hist_filt.sort_values(['futu_key', 'Date'], ascending=[True, False])

# Turnover (Mean 30d)
turnover_feat = df_hist_filt.groupby('futu_key').head(30).groupby('futu_key')['Turnover'].mean().reset_index()
turnover_feat.rename(columns={'Turnover': 'avg_turnover'}, inplace=True)

# Price (Latest)
price_feat = df_hist_filt.groupby('futu_key').head(1)[['futu_key', 'Close']]

# Shares & Float (Drop duplicates to avoid join errors)
shares_feat = df_shares.drop_duplicates(subset=['futu_key'])[['futu_key', 'Total Shares', 'Float %']].copy()

# --- C. Clean Float % ---
# Ensure Float % is numeric (handle potential missing values)
shares_feat['Float %'] = pd.to_numeric(shares_feat['Float %'], errors='coerce').fillna(0)

# Merge Price + Shares to get Cap
cap_df = pd.merge(price_feat, shares_feat, on='futu_key', how='inner')
cap_df['market_cap'] = cap_df['Close'] * cap_df['Total Shares']

# --- D. Master Table ---
# Merge Target + Basic Info + Calculated Features
df_master = pd.merge(df_target, df_basic, on='code', how='left')
df_master = pd.merge(df_master, turnover_feat, left_on='code', right_on='futu_key', how='inner')
df_master = pd.merge(df_master, cap_df, on='futu_key', how='inner')

# --- E. Hard Filters (Pink & New) ---
# We strictly filter these OUT before training, as discussed
pink_exchanges = ['US_PINK', 'PINK', 'OTC'] 
df_master['is_pink'] = df_master['exchange_type'].isin(pink_exchanges)
df_master['listing_date'] = pd.to_datetime(df_master['listing_date'], errors='coerce')
df_master['days_listed'] = (ANCHOR_DATE - df_master['listing_date']).dt.days
df_master['is_new'] = df_master['days_listed'] < 90

# TRAIN SET (Eligible stocks only)
df_train = df_master[(~df_master['is_pink']) & (~df_master['is_new'])].copy()
target_col = 'is_long_permit'
y = df_train[target_col].astype(int)

print(f"Training Universe: {len(df_train)} stocks")

# ==========================================
# 2. DEFINING THE 3 EXPERIMENTS
# ==========================================
results = []

def run_experiment(name, X_data):
    # Split
    X_train, X_test, y_train, y_test = train_test_split(X_data, y, test_size=0.2, random_state=42)
    
    # Train
    # Increase max_iter for linear scaling models to ensure convergence
    model = LogisticRegression(max_iter=1000) 
    model.fit(X_train, y_train)
    
    # Evaluate
    probs = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, probs)
    
    # Check FPR at threshold 0.65
    preds = (probs >= 0.65).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    fpr = fp / (tn + fp) if (tn + fp) > 0 else 0
    accuracy = accuracy_score(y_test, preds)
    
    return {
        'Experiment': name,
        'AUC': auc,
        'Accuracy': accuracy,
        'FPR (Risk)': fpr,
        'Coefficients': model.coef_[0]
    }

print("\nRunning Tournament...")

# --- EXPERIMENT 1: LOG TRANSFORM + FLOAT % ---
# Features: Log(Turnover), Log(Cap), Float %
df_exp1 = pd.DataFrame()
df_exp1['log_turnover'] = np.log1p(df_train['avg_turnover'])
df_exp1['log_cap'] = np.log1p(df_train['market_cap'])
df_exp1['float_pct'] = df_train['Float %']

res1 = run_experiment("1. Log + Float%", df_exp1)
results.append(res1)


# --- EXPERIMENT 2: STANDARD SCALER (Regularized Linear) ---
# Features: Turnover (Scaled), Cap (Scaled)
# Note: We scale the WHOLE dataset first for simplicity in this snippet, 
# strictly one should fit scaler on train and transform test.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df_train[['avg_turnover', 'market_cap']])
df_exp2 = pd.DataFrame(scaled_features, columns=['sc_turnover', 'sc_cap'])

res2 = run_experiment("2. Std Scaler (Linear)", df_exp2)
results.append(res2)


# --- EXPERIMENT 3: CUSTOM SCALING (/1M, /1B) ---
# Features: Turnover/1M, Cap/1B
df_exp3 = pd.DataFrame()
df_exp3['turnover_1M'] = df_train['avg_turnover'] / 1_000_000
df_exp3['cap_1B'] = df_train['market_cap'] / 1_000_000_000

res3 = run_experiment("3. Custom /1M /1B", df_exp3)
results.append(res3)

# ==========================================
# ADDING EXPERIMENT 4: MARKET CAP BINS
# ==========================================

print("\n--- EXPERIMENT 4: BINNED MARKET CAP ---")

# 1. Define the Bin Logic
def assign_bin(cap_value):
    # Convert cap to Billions for easier reading
    cap_B = cap_value / 1_000_000_000
    
    if cap_B > 20: return 'Bin_1_GT_20B'
    if cap_B > 10: return 'Bin_2_10B_to_20B'
    if cap_B > 5:  return 'Bin_3_5B_to_10B'
    if cap_B > 3:  return 'Bin_4_3B_to_5B'
    if cap_B > 1:  return 'Bin_5_1B_to_3B'
    if cap_B > 0.5: return 'Bin_6_0.5B_to_1B'
    return 'Bin_7_LT_0.5B'

# 2. Create the Binned DataFrame
df_exp4 = pd.DataFrame()

# We still use Log Turnover (as it was the best liquidity feature) and Float %
df_exp4['log_turnover'] = np.log1p(df_train['avg_turnover'])
df_exp4['float_pct'] = df_train['Float %']

# Apply Binning
# We create a categorical column first
bins = df_train['market_cap'].apply(assign_bin)

# One-Hot Encode the Bins (Create a True/False column for each bin)
# prefix='' removes the column name prefix for cleaner reading
bins_dummies = pd.get_dummies(bins, prefix='Cap') 

# Concatenate: [Log Turnover, Float, Cap_Bin_1, Cap_Bin_2, ...]
df_exp4 = pd.concat([df_exp4, bins_dummies], axis=1)


# ==========================================
# 3. LEADERBOARD & ANALYSIS
# ==========================================
res_df = pd.DataFrame(results).set_index('Experiment')

print("\n" + "="*40)
print("TOURNAMENT RESULTS")
print("="*40)
print(res_df[['AUC', 'Accuracy', 'FPR (Risk)']])

print("\n--- Detailed Coefficients ---")
for res in results:
    print(f"\n> {res['Experiment']}:")
    print(f"  {res['Coefficients']}")

best_model = res_df['AUC'].idxmax()
print(f"\n🏆 WINNER: {best_model}")

# Optional: Suggestion based on winner
if "Log" in best_model:
    print("Insight: The Log transform handles the massive difference between 'Mega Caps' and 'Small Caps' best.")
else:
    print("Insight: Linear scaling proved effective, suggesting the raw magnitude matters directly.")

Loading and preparing data...
Training Universe: 3941 stocks

Running Tournament...

--- EXPERIMENT 4: BINNED MARKET CAP ---

TOURNAMENT RESULTS
                             AUC  Accuracy  FPR (Risk)
Experiment                                            
1. Log + Float%         0.910869  0.807351    0.127219
2. Std Scaler (Linear)  0.888735  0.680608    0.044379
3. Custom /1M /1B       0.884517  0.709759    0.068047

--- Detailed Coefficients ---

> 1. Log + Float%:
  [0.53030448 0.65935815 0.01798502]

> 2. Std Scaler (Linear):
  [10.43925413  4.4361197 ]

> 3. Custom /1M /1B:
  [0.02203298 0.023424  ]

🏆 WINNER: 1. Log + Float%
Insight: The Log transform handles the massive difference between 'Mega Caps' and 'Small Caps' best.


In [9]:
# 3. Run Experiment 4
res4 = run_experiment("4. Log Turn + Float + Binned Cap", df_exp4)
results.append(res4)

# ==========================================
# UPDATED LEADERBOARD
# ==========================================
res_df_updated = pd.DataFrame(results).set_index('Experiment')

print("\n" + "="*40)
print("UPDATED TOURNAMENT RESULTS (With Binning)")
print("="*40)
print(res_df_updated[['AUC', 'Accuracy', 'FPR (Risk)']])

print("\n--- Detailed Coefficients (Exp 4) ---")
# Mapping coefficients to column names for clarity
feature_names = df_exp4.columns.tolist()
coeffs = res4['Coefficients']
for feat, coef in zip(feature_names, coeffs):
    print(f"{feat:.<25} {coef:.4f}")

best_model = res_df_updated['AUC'].idxmax()
print(f"\n🏆 CURRENT CHAMPION: {best_model}")


UPDATED TOURNAMENT RESULTS (With Binning)
                                       AUC  Accuracy  FPR (Risk)
Experiment                                                      
1. Log + Float%                   0.910869  0.807351    0.127219
2. Std Scaler (Linear)            0.888735  0.680608    0.044379
3. Custom /1M /1B                 0.884517  0.709759    0.068047
4. Log Turn + Float + Binned Cap  0.910081  0.812421    0.124260

--- Detailed Coefficients (Exp 4) ---
log_turnover............. 0.5680
float_pct................ 0.0177
Cap_Bin_1_GT_20B......... 1.2172
Cap_Bin_2_10B_to_20B..... 0.9172
Cap_Bin_3_5B_to_10B...... 0.5181
Cap_Bin_4_3B_to_5B....... 0.3583
Cap_Bin_5_1B_to_3B....... -0.2551
Cap_Bin_6_0.5B_to_1B..... -1.0841
Cap_Bin_7_LT_0.5B........ -1.7068

🏆 CURRENT CHAMPION: 1. Log + Float%


In [10]:
# ==========================================
# ADDING EXPERIMENT 0: THE BASELINE (Control Group)
# ==========================================
print("\n--- EXPERIMENT 0: BASELINE (Log Only) ---")

df_exp0 = pd.DataFrame()
df_exp0['log_turnover'] = np.log1p(df_train['avg_turnover'])
df_exp0['log_cap'] = np.log1p(df_train['market_cap'])

# Run the Baseline
res0 = run_experiment("0. Baseline (Log Only)", df_exp0)

# Prepend to results (insert at the top for easy comparison)
results.insert(0, res0)

# ==========================================
# FINAL TOURNAMENT STANDINGS
# ==========================================
res_df_final = pd.DataFrame(results).set_index('Experiment')

# Sort by AUC to show the true rank
res_df_final = res_df_final.sort_values(by='AUC', ascending=False)

print("\n" + "="*50)
print("FINAL LEADERBOARD (Sorted by Predictive Power)")
print("="*50)
print(res_df_final[['AUC', 'Accuracy', 'FPR (Risk)']])

# --- CRITICAL COMPARISON ---
# Calculate the "Lift" provided by the new features
baseline_auc = res0['AUC']
best_auc = res_df_final.iloc[0]['AUC']
winner_name = res_df_final.index[0]
lift = (best_auc - baseline_auc) / baseline_auc

print("\n--- CONCLUSION ---")
print(f"The Baseline (Log Only) AUC: {baseline_auc:.5f}")
print(f"The Winner ({winner_name}) AUC: {best_auc:.5f}")

if winner_name == "0. Baseline (Log Only)":
    print("👉 VERDICT: The original simple model is best. Adding Float% or Bins creates noise.")
elif lift < 0.005: # Less than 0.5% improvement
    print(f"👉 VERDICT: Improvement is marginal (+{lift:.2%}). Stick with Baseline for simplicity.")
else:
    print(f"👉 VERDICT: Significant improvement (+{lift:.2%}). The extra complexity is justified.")


--- EXPERIMENT 0: BASELINE (Log Only) ---

FINAL LEADERBOARD (Sorted by Predictive Power)
                                       AUC  Accuracy  FPR (Risk)
Experiment                                                      
1. Log + Float%                   0.910869  0.807351    0.127219
4. Log Turn + Float + Binned Cap  0.910081  0.812421    0.124260
0. Baseline (Log Only)            0.908310  0.797212    0.109467
2. Std Scaler (Linear)            0.888735  0.680608    0.044379
3. Custom /1M /1B                 0.884517  0.709759    0.068047

--- CONCLUSION ---
The Baseline (Log Only) AUC: 0.90831
The Winner (1. Log + Float%) AUC: 0.91087
👉 VERDICT: Improvement is marginal (+0.28%). Stick with Baseline for simplicity.


In [11]:
# ==========================================
# EXPERIMENT 6: THE HYBRID (Log Cap/Turnover + Binned Float)
# ==========================================
print("\n" + "="*50)
print("FLOAT BATTLE: LINEAR vs. BINNED")
print("="*50)

# --- 1. DEFINE THE COMPETITORS ---

# MODEL A: Log + Linear Float (The previous winner?)
df_model_A = pd.DataFrame()
df_model_A['log_turnover'] = np.log1p(df_train['avg_turnover'])
df_model_A['log_cap'] = np.log1p(df_train['market_cap'])
df_model_A['float_pct'] = df_train['Float %']

# MODEL B: Log + Binned Float (The Challenger)
df_model_B = pd.DataFrame()
df_model_B['log_turnover'] = np.log1p(df_train['avg_turnover'])
df_model_B['log_cap'] = np.log1p(df_train['market_cap'])

# Bin the Float into 5% buckets (0-5, 5-10, ... 95-100)
float_bins_range = range(0, 105, 5) 
labels = [f"Float_{i}-{i+5}" for i in range(0, 100, 5)]
float_cat = pd.cut(df_train['Float %'], bins=float_bins_range, labels=labels, include_lowest=True)
float_dummies = pd.get_dummies(float_cat)

# Combine Log features with the Float Bins
df_model_B = pd.concat([df_model_B, float_dummies], axis=1)

# --- 2. RUN HEAD-TO-HEAD ---
res_A = run_experiment("Model A: Linear Float", df_model_A)
res_B = run_experiment("Model B: Binned Float (5%)", df_model_B)

# --- 3. PRINT RESULTS ---
print(f"Model A (Linear) AUC: {res_A['AUC']:.5f}")
print(f"Model B (Binned) AUC: {res_B['AUC']:.5f}")

diff = res_B['AUC'] - res_A['AUC']
if diff > 0.001:
    print(f"\n✅ WINNER: Binned Float (+{diff:.5f})")
    print("Insight: There are specific 'Cliff Edges' in the float percentage that matter.")
    
    # Analyze WHICH bins matter
    print("\n--- Significant Float Zones ---")
    coefs = res_B['Coefficients']
    cols = df_model_B.columns
    
    # Filter for Float bins with strong positive impact
    bin_impact = []
    for c, val in zip(cols, coefs):
        if "Float_" in c:
            bin_impact.append((c, val))
    
    # Sort by impact
    bin_impact.sort(key=lambda x: x[1], reverse=True)
    
    print("Top 3 'Safe Zones' (Highest Positive Coeffs):")
    for name, val in bin_impact[:3]:
        print(f"  {name}: {val:.4f}")
        
    print("\nTop 3 'Kill Zones' (Lowest Negative Coeffs):")
    for name, val in bin_impact[-3:]:
        print(f"  {name}: {val:.4f}")
        
else:
    print(f"\n❌ WINNER: Linear Float (Difference {diff:.5f} is negligible)")
    print("Insight: Complexity didn't pay off. Just knowing 'More Float is Better' is enough.")


FLOAT BATTLE: LINEAR vs. BINNED
Model A (Linear) AUC: 0.91087
Model B (Binned) AUC: 0.91017

❌ WINNER: Linear Float (Difference -0.00070 is negligible)
Insight: Complexity didn't pay off. Just knowing 'More Float is Better' is enough.


In [12]:
# ==========================================
# ADDING EXPERIMENT 5: BINNED FLOAT (5% Bins)
# ==========================================
print("\n--- EXPERIMENT 5: BINNED FLOAT (5% Intervals) ---")

df_exp5 = pd.DataFrame()

# 1. Base Features (Baseline Log)
df_exp5['log_turnover'] = np.log1p(df_train['avg_turnover'])
df_exp5['log_cap'] = np.log1p(df_train['market_cap'])

# 2. Bin the Float
# Create bins from 0 to 100 with step 5
float_bins_range = range(0, 105, 5) 
labels = [f"Float_{i}-{i+5}" for i in range(0, 100, 5)]
float_cat = pd.cut(df_train['Float %'], bins=float_bins_range, labels=labels, include_lowest=True)
float_dummies = pd.get_dummies(float_cat)

# 3. Combine
df_exp5 = pd.concat([df_exp5, float_dummies], axis=1)

# 4. Run Experiment
res5 = run_experiment("5. Log + Binned Float (5%)", df_exp5)
results.append(res5)

# ==========================================
# FINAL LEADERBOARD (ALL MODELS)
# ==========================================
res_df_final = pd.DataFrame(results).set_index('Experiment')

# Sort by AUC to show the true rank
res_df_final = res_df_final.sort_values(by='AUC', ascending=False)

print("\n" + "="*80)
print("FINAL COMPLETE LEADERBOARD")
print("="*80)
# Showing all columns including Accuracy
print(res_df_final[['AUC', 'Accuracy', 'FPR (Risk)']])

# --- CRITICAL COMPARISON ---
baseline_auc = res_df_final.loc["0. Baseline (Log Only)"]['AUC']
best_model_name = res_df_final.index[0]
best_auc = res_df_final.iloc[0]['AUC']

lift = (best_auc - baseline_auc) / baseline_auc

print("\n--- CONCLUSION ---")
print(f"Baseline (Log Only) AUC: {baseline_auc:.5f}")
print(f"Winner ({best_model_name}) AUC:   {best_auc:.5f}")
print(f"Lift: {lift:+.2%}")

if lift < 0.005:
    print("👉 VERDICT: Improvement is marginal (< 0.5%). Stick with Baseline.")
else:
    print("👉 VERDICT: Significant improvement. The winning feature engineering is worth keeping.")


--- EXPERIMENT 5: BINNED FLOAT (5% Intervals) ---

FINAL COMPLETE LEADERBOARD
                                       AUC  Accuracy  FPR (Risk)
Experiment                                                      
1. Log + Float%                   0.910869  0.807351    0.127219
5. Log + Binned Float (5%)        0.910167  0.806084    0.124260
4. Log Turn + Float + Binned Cap  0.910081  0.812421    0.124260
0. Baseline (Log Only)            0.908310  0.797212    0.109467
2. Std Scaler (Linear)            0.888735  0.680608    0.044379
3. Custom /1M /1B                 0.884517  0.709759    0.068047

--- CONCLUSION ---
Baseline (Log Only) AUC: 0.90831
Winner (1. Log + Float%) AUC:   0.91087
Lift: +0.28%
👉 VERDICT: Improvement is marginal (< 0.5%). Stick with Baseline.
